In [11]:
import pandas as pd
import numpy as np


# 1. LOAD DATA & FIX MISSING VALUES

df = pd.read_csv('Price_Dataset.csv')
df = df.sort_values('year').reset_index(drop=True)

# Interpolate missing USD index (2020-2024) using linear extrapolation
df['usd_index'] = df['usd_index'].interpolate(method='linear', fill_value='extrapolate', limit_direction='forward')
# If interpolate doesn't extrapolate, use forward fill of last known trend
if df['usd_index'].isnull().any():
    last_known_idx = df['usd_index'].last_valid_index()
    last_val = df.loc[last_known_idx, 'usd_index']
    # Use the average annual change from last 5 known years to project
    recent = df.loc[last_known_idx-4:last_known_idx, 'usd_index']
    avg_change = recent.diff().mean()
    for i in range(last_known_idx+1, len(df)):
        df.loc[i, 'usd_index'] = last_val + avg_change * (i - last_known_idx)

print("Missing values after fix:")
print(df.isnull().sum())
print()

# 2. LOG TRANSFORMS
log_cols = [
    'pistachio_price_lb', 'pistachio_production_mt', 'pistachio_bearing_acres',
    'iran_production_mt', 'almond_price_lb', 'SP500', 'CPI',
    'us_disposable_income_bn', 'china_gdp_trillion_usd',
    'wti_oil_price', 'fertilizer_ppi'
]

for col in log_cols:
    df[f'log_{col}'] = np.log(df[col])

print("Log transforms created for:", log_cols)
print()


# 3. FIRST DIFFERENCES OF LOG VARIABLES (≈ % changes)

log_columns = [c for c in df.columns if c.startswith('log_')]
for col in log_columns:
    df[f'd_{col}'] = df[col].diff()

print("First differences created for all log variables")
print()


# 4. LAGGED VARIABLES (1-year and 2-year)

lag_cols = [
    'log_pistachio_price_lb', 'log_pistachio_production_mt',
    'log_iran_production_mt', 'log_wti_oil_price',
    'log_fertilizer_ppi', 'fresno_rainfall_inches'
]

for col in lag_cols:
    df[f'{col}_lag1'] = df[col].shift(1)
    df[f'{col}_lag2'] = df[col].shift(2)

print("Lags (1 and 2 year) created for:", lag_cols)
print()


# 5. ROLLING STATISTICS (3-year window)

roll_cols = [
    'log_pistachio_price_lb', 'log_pistachio_production_mt',
    'fresno_rainfall_inches', 'log_wti_oil_price'
]

for col in roll_cols:
    df[f'{col}_roll3_mean'] = df[col].rolling(window=3, min_periods=2).mean()
    df[f'{col}_roll3_std']  = df[col].rolling(window=3, min_periods=2).std()

print("Rolling 3-year mean and std created for:", roll_cols)
print()


# 6. INTERACTION TERMS

# Global supply pressure: US production * Iran production
df['interact_us_iran_prod'] = df['log_pistachio_production_mt'] * df['log_iran_production_mt']

# Weather-adjusted supply: US production * rainfall
df['interact_prod_rainfall'] = df['log_pistachio_production_mt'] * df['fresno_rainfall_inches']

# Combined input cost: oil * fertilizer
df['interact_oil_fert'] = df['log_wti_oil_price'] * df['log_fertilizer_ppi']

print("Interaction terms created: us_iran_prod, prod_rainfall, oil_fert")
print()


# 7. BIENNIAL BEARING DUMMY

df['production_change'] = df['pistachio_production_mt'].diff()
df['biennial_dummy'] = (df['production_change'] > 0).astype(int)

print("Biennial dummy: 1 = production increased (on-year), 0 = decreased (off-year)")
print()

# 8. REAL PRICE (inflation-adjusted)

base_cpi = df.loc[df['year'] == 1980, 'CPI'].values[0]
df['real_price'] = df['pistachio_price_lb'] / (df['CPI'] / base_cpi)
df['log_real_price'] = np.log(df['real_price'])

print("Real price computed using 1980 as base year")
print()


# 9. YIELD PER ACRE

df['yield_per_acre'] = df['pistachio_production_mt'] / df['pistachio_bearing_acres']
df['log_yield_per_acre'] = np.log(df['yield_per_acre'])

print("Yield per acre = production / bearing acres")
print()


# 10. TRAIN-TEST SPLIT (80/20 by time)

split_idx = int(len(df) * 0.8)  # should be ~36
train = df.iloc[:split_idx].copy()
test  = df.iloc[split_idx:].copy()

print(f"Split index: {split_idx}")
print(f"Train: {train['year'].min()}-{train['year'].max()} ({len(train)} obs)")
print(f"Test:  {test['year'].min()}-{test['year'].max()} ({len(test)} obs)")
print()


# SUMMARY: print all columns

print(f"Total columns in engineered dataset: {len(df.columns)}")
print()
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col}")

print()
print("=== First 5 rows of key engineered features ===")
key_cols = ['year', 'log_pistachio_price_lb', 'd_log_pistachio_price_lb',
            'log_pistachio_production_mt_lag1', 'interact_us_iran_prod',
            'biennial_dummy', 'real_price', 'yield_per_acre']
print(df[key_cols].head(10).to_string())

Missing values after fix:
year                       0
pistachio_price_lb         0
pistachio_bearing_acres    0
pistachio_production_mt    0
almond_price_lb            0
iran_production_mt         0
SP500                      0
CPI                        0
wti_oil_price              0
electricity_price_kwh      0
fertilizer_ppi             0
us_disposable_income_bn    0
china_gdp_trillion_usd     0
fresno_rainfall_inches     0
usd_index                  0
dtype: int64

Log transforms created for: ['pistachio_price_lb', 'pistachio_production_mt', 'pistachio_bearing_acres', 'iran_production_mt', 'almond_price_lb', 'SP500', 'CPI', 'us_disposable_income_bn', 'china_gdp_trillion_usd', 'wti_oil_price', 'fertilizer_ppi']

First differences created for all log variables

Lags (1 and 2 year) created for: ['log_pistachio_price_lb', 'log_pistachio_production_mt', 'log_iran_production_mt', 'log_wti_oil_price', 'log_fertilizer_ppi', 'fresno_rainfall_inches']

Rolling 3-year mean and std created fo

In [12]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, het_arch
from scipy import stats
import warnings
warnings.filterwarnings('ignore')


# STEP 1: UNIT ROOT TESTS 

test_vars = [
    'log_pistachio_price_lb', 'log_pistachio_production_mt',
    'log_wti_oil_price', 'log_CPI', 'fresno_rainfall_inches'
]

print("=" * 80)
print("UNIT ROOT TESTS (reduced variable set)")
print("=" * 80)
print(f"{'Variable':<40} {'ADF stat':>10} {'ADF p':>8} {'KPSS stat':>10} {'KPSS p':>8} {'Order':>6}")
print("-" * 80)

integration_orders = {}

for var in test_vars:
    series = df[var].dropna()
    adf_stat, adf_p, _, _, _, _ = adfuller(series, autolag='AIC')
    kpss_stat, kpss_p, _, _ = kpss(series, regression='c', nlags='auto')

    if adf_p < 0.05 and kpss_p > 0.05:
        order = 'I(0)'
    elif adf_p >= 0.05 and kpss_p <= 0.05:
        order = 'I(1)'
    elif adf_p < 0.05 and kpss_p <= 0.05:
        order = 'I(0)*'
    else:
        order = 'I(1)*'

    integration_orders[var] = order
    print(f"{var:<40} {adf_stat:>10.4f} {adf_p:>8.4f} {kpss_stat:>10.4f} {kpss_p:>8.4f} {order:>6}")

print()


# STEP 2: BUILD ARDL ECM WITH 4 VARIABLES, MAX LAG 1

y_var = 'log_pistachio_price_lb'

x_vars = [
    'log_pistachio_production_mt',
    'log_wti_oil_price',
    'log_CPI',
    'fresno_rainfall_inches'
]

max_lag = 1

ardl_df = df[['year', y_var] + x_vars].dropna().copy()

# Dependent variable lag and difference
ardl_df[f'{y_var}_lag1'] = ardl_df[y_var].shift(1)
ardl_df[f'd_{y_var}'] = ardl_df[y_var].diff()

# Independent variable diffs, lags, diff-lags
for var in x_vars:
    ardl_df[f'd_{var}'] = ardl_df[var].diff()
    ardl_df[f'{var}_lag1'] = ardl_df[var].shift(1)
    ardl_df[f'd_{var}_lag1'] = ardl_df[f'd_{var}'].shift(1)

# Lagged diff of dependent
ardl_df[f'd_{y_var}_lag1'] = ardl_df[f'd_{y_var}'].shift(1)

# Drop NaN rows
ardl_df = ardl_df.dropna().reset_index(drop=True)

# ECM regressor names
ecm_x_names = []

# Short-run: lagged diff of y
ecm_x_names.append(f'd_{y_var}_lag1')

# Short-run: current diff and lagged diff of each x
for var in x_vars:
    ecm_x_names.append(f'd_{var}')
    ecm_x_names.append(f'd_{var}_lag1')

# Long-run: lagged levels
ecm_x_names.append(f'{y_var}_lag1')
for var in x_vars:
    ecm_x_names.append(f'{var}_lag1')

ecm_y = ardl_df[f'd_{y_var}']
ecm_X = add_constant(ardl_df[ecm_x_names].astype(float))

n_params = len(ecm_x_names) + 1  # +1 for constant
print(f"ARDL dataset: {len(ardl_df)} observations")
print(f"Parameters: {n_params}")
print(f"Degrees of freedom: {len(ardl_df) - n_params}")
print(f"Year range: {ardl_df['year'].min()} - {ardl_df['year'].max()}")
print()

# Estimate ECM
ecm_model = OLS(ecm_y.astype(float), ecm_X).fit()
print("=" * 80)
print("ARDL ERROR CORRECTION MODEL — REVISED")
print("=" * 80)
print(ecm_model.summary())
print()


# STEP 3: BOUNDS TEST

level_vars = [f'{y_var}_lag1'] + [f'{var}_lag1' for var in x_vars]

restricted_x_names = [n for n in ecm_x_names if n not in level_vars]
restricted_X = add_constant(ardl_df[restricted_x_names].astype(float))
restricted_model = OLS(ecm_y.astype(float), restricted_X).fit()

k = len(level_vars)
rss_r = restricted_model.ssr
rss_u = ecm_model.ssr
df_u = ecm_model.df_resid

f_stat = ((rss_r - rss_u) / k) / (rss_u / df_u)

print("=" * 80)
print("PESARAN BOUNDS TEST FOR COINTEGRATION")
print("=" * 80)
print(f"F-statistic: {f_stat:.4f}")
print(f"Number of level variables (k): {k}")
print()
print("Critical values (Pesaran et al. 2001, Case III):")
print("  k=4 variables:")
print("  Significance   I(0) bound   I(1) bound")
print("  10%            2.45         3.52")
print("  5%             2.86         4.01")
print("  1%             3.74         5.06")
print()
if f_stat > 5.06:
    print(f"  RESULT: F={f_stat:.4f} > 5.06 → COINTEGRATION at 1% level")
elif f_stat > 4.01:
    print(f"  RESULT: F={f_stat:.4f} > 4.01 → COINTEGRATION at 5% level")
elif f_stat > 3.52:
    print(f"  RESULT: F={f_stat:.4f} > 3.52 → COINTEGRATION at 10% level")
elif f_stat < 2.45:
    print(f"  RESULT: F={f_stat:.4f} < 2.45 → NO COINTEGRATION")
else:
    print(f"  RESULT: F={f_stat:.4f} is between bounds → INCONCLUSIVE")
print()


# STEP 4: LONG-RUN COEFFICIENTS

print("=" * 80)
print("LONG-RUN COEFFICIENTS")
print("=" * 80)

ect_coeff = ecm_model.params[f'{y_var}_lag1']
ect_pval  = ecm_model.pvalues[f'{y_var}_lag1']

print(f"Error Correction Term: {ect_coeff:.4f} (p={ect_pval:.4f})")
if -1 < ect_coeff < 0:
    print(f"  → {abs(ect_coeff)*100:.1f}% of disequilibrium corrected per year (valid)")
elif ect_coeff < -1:
    print(f"  → Overshooting adjustment ({abs(ect_coeff)*100:.1f}%) — may indicate instability")
else:
    print(f"  ⚠ ECT is positive or zero — model may be misspecified")
print()

print(f"{'Variable':<40} {'LR Coeff':>10} {'p-value':>10} {'Sig':>5}")
print("-" * 70)
for var in x_vars:
    var_coeff = ecm_model.params[f'{var}_lag1']
    var_pval  = ecm_model.pvalues[f'{var}_lag1']
    lr_coeff  = -var_coeff / ect_coeff if ect_coeff != 0 else np.nan
    sig = '***' if var_pval < 0.01 else '**' if var_pval < 0.05 else '*' if var_pval < 0.1 else ''
    print(f"{var:<40} {lr_coeff:>10.4f} {var_pval:>10.4f} {sig:>5}")

print()


# STEP 5: SHORT-RUN COEFFICIENTS

print("=" * 80)
print("SHORT-RUN COEFFICIENTS")
print("=" * 80)
sr_vars = [f'd_{y_var}_lag1'] + [f'd_{var}' for var in x_vars] + [f'd_{var}_lag1' for var in x_vars]
print(f"{'Variable':<45} {'Coeff':>10} {'p-value':>10} {'Sig':>5}")
print("-" * 75)
for var in sr_vars:
    coeff = ecm_model.params[var]
    pval  = ecm_model.pvalues[var]
    sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
    print(f"{var:<45} {coeff:>10.4f} {pval:>10.4f} {sig:>5}")
print()


# STEP 6: DIAGNOSTICS

print("=" * 80)
print("DIAGNOSTIC TESTS")
print("=" * 80)

residuals = ecm_model.resid

# Breusch-Godfrey
try:
    bg_stat, bg_pval, _, _ = acorr_breusch_godfrey(ecm_model, nlags=2)
    print(f"Breusch-Godfrey (serial correlation):")
    print(f"  LM={bg_stat:.4f}, p={bg_pval:.4f}")
    print(f"  {'→ No serial correlation' if bg_pval > 0.05 else '→ Serial correlation detected'}")
except Exception as e:
    print(f"Breusch-Godfrey failed: {e}")
print()

# ARCH-LM
try:
    arch_stat, arch_pval, _, _ = het_arch(residuals, nlags=2)
    print(f"ARCH-LM (heteroskedasticity):")
    print(f"  LM={arch_stat:.4f}, p={arch_pval:.4f}")
    print(f"  {'→ No ARCH effects' if arch_pval > 0.05 else '→ ARCH effects present'}")
except Exception as e:
    print(f"ARCH-LM failed: {e}")
print()

# Jarque-Bera
jb_stat, jb_pval = stats.jarque_bera(residuals)
print(f"Jarque-Bera (normality):")
print(f"  JB={jb_stat:.4f}, p={jb_pval:.4f}")
print(f"  {'→ Residuals normal' if jb_pval > 0.05 else '→ Residuals not normal'}")
print()

# Durbin-Watson
print(f"R-squared:     {ecm_model.rsquared:.4f}")
print(f"Adj R-squared: {ecm_model.rsquared_adj:.4f}")
print(f"Durbin-Watson: {ecm_model.summary2().tables[0].to_dict()[1].get('Durbin-Watson:', 'N/A') if hasattr(ecm_model, 'summary2') else 'see summary'}")
print()

# CUSUM
print("CUSUM TEST:")
try:
    from statsmodels.stats.diagnostic import recursive_olsresiduals
    rr = recursive_olsresiduals(ecm_model)
    rec_resid = rr[3]
    cusum = np.cumsum(rec_resid) / np.std(rec_resid)
    n_rr = len(rec_resid)
    upper_bound = 0.948 * np.sqrt(n_rr + 2 * np.arange(1, n_rr + 1) / n_rr)
    max_cusum = np.max(np.abs(cusum))
    max_bound = np.max(upper_bound)
    print(f"  Max |CUSUM| = {max_cusum:.4f}, boundary = {max_bound:.4f}")
    print(f"  {'→ STABLE' if max_cusum < max_bound else '→ UNSTABLE'}")
except Exception as e:
    print(f"  CUSUM failed: {e}")

print()
print("=" * 80)
print("ARDL DONE")
print("=" * 80)

UNIT ROOT TESTS (reduced variable set)
Variable                                   ADF stat    ADF p  KPSS stat   KPSS p  Order
--------------------------------------------------------------------------------
log_pistachio_price_lb                      -2.2445   0.1905     0.6453   0.0185   I(1)
log_pistachio_production_mt                 -1.2669   0.6442     0.9906   0.0100   I(1)
log_wti_oil_price                           -1.1180   0.7079     0.6759   0.0157   I(1)
log_CPI                                     -1.1611   0.6901     1.0052   0.0100   I(1)
fresno_rainfall_inches                      -6.4823   0.0000     0.1913   0.1000   I(0)

ARDL dataset: 43 observations
Parameters: 15
Degrees of freedom: 28
Year range: 1982 - 2024

ARDL ERROR CORRECTION MODEL — REVISED
                               OLS Regression Results                               
Dep. Variable:     d_log_pistachio_price_lb   R-squared:                       0.631
Model:                                  OLS   Adj.

In [13]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


# STEP 1: REDUCED FEATURE SET

target = 'log_pistachio_price_lb'

feature_cols = [
    'log_pistachio_production_mt_lag1',  # #1 consensus driver — last year's supply
    'log_wti_oil_price',                 # #2 consensus — input costs
    'interact_oil_fert',                 # #3 consensus — combined input cost pressure
    'log_us_disposable_income_bn',       # #4 consensus — demand-side purchasing power
    'log_almond_price_lb',              # substitute product price
    'fresno_rainfall_inches',            # weather shock
    'log_pistachio_price_lb_lag1',       # price persistence / mean reversion
    'biennial_dummy',                    # agronomic cycle
]

feature_cols = [c for c in feature_cols if c in df.columns]
print(f"Features selected: {len(feature_cols)}")
print(f"Features: {feature_cols}")
print()

model_df = df[['year', target] + feature_cols].dropna().reset_index(drop=True)
print(f"Usable observations: {len(model_df)}")
print(f"Year range: {model_df['year'].min()} - {model_df['year'].max()}")
print()


# STEP 2: TRAIN-TEST SPLIT

split_idx = int(len(model_df) * 0.8)

train_df = model_df.iloc[:split_idx]
test_df  = model_df.iloc[split_idx:]

X_train = train_df[feature_cols]
y_train = train_df[target]
X_test  = test_df[feature_cols]
y_test  = test_df[target]

print(f"Train: {train_df['year'].min()}-{train_df['year'].max()} ({len(train_df)} obs)")
print(f"Test:  {test_df['year'].min()}-{test_df['year'].max()} ({len(test_df)} obs)")
print(f"Feature-to-observation ratio: {len(feature_cols)}:{len(train_df)} = 1:{len(train_df)//len(feature_cols)}")
print()


# STEP 3: RANDOM FOREST (tighter regularization)

print("=" * 80)
print("RANDOM FOREST")
print("=" * 80)

tscv = TimeSeriesSplit(n_splits=3)

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=3,            # shallower than before
    min_samples_split=5,
    min_samples_leaf=4,     # stricter leaf size
    max_features=0.6,       # use 60% of features per split
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_train_pred = rf.predict(X_train)
rf_test_pred  = rf.predict(X_test)

rf_train_r2   = r2_score(y_train, rf_train_pred)
rf_test_r2    = r2_score(y_test, rf_test_pred)
rf_train_rmse = np.sqrt(mean_squared_error(y_train, rf_train_pred))
rf_test_rmse  = np.sqrt(mean_squared_error(y_test, rf_test_pred))
rf_train_mae  = mean_absolute_error(y_train, rf_train_pred)
rf_test_mae   = mean_absolute_error(y_test, rf_test_pred)

print(f"{'Metric':<20} {'Train':>10} {'Test':>10}")
print("-" * 42)
print(f"{'R²':<20} {rf_train_r2:>10.4f} {rf_test_r2:>10.4f}")
print(f"{'RMSE':<20} {rf_train_rmse:>10.4f} {rf_test_rmse:>10.4f}")
print(f"{'MAE':<20} {rf_train_mae:>10.4f} {rf_test_mae:>10.4f}")
print()

cv_scores = cross_val_score(rf, X_train, y_train, cv=tscv, scoring='r2')
print(f"CV R² scores: {[f'{s:.4f}' for s in cv_scores]}")
print(f"Mean CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print()


# STEP 4: GRADIENT BOOSTING (tighter regularization)

print("=" * 80)
print("GRADIENT BOOSTING")
print("=" * 80)

gbt = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=2,            # very shallow
    learning_rate=0.03,     # slower learning
    min_samples_split=5,
    min_samples_leaf=4,
    subsample=0.7,
    random_state=42
)

gbt.fit(X_train, y_train)

gbt_train_pred = gbt.predict(X_train)
gbt_test_pred  = gbt.predict(X_test)

gbt_train_r2   = r2_score(y_train, gbt_train_pred)
gbt_test_r2    = r2_score(y_test, gbt_test_pred)
gbt_train_rmse = np.sqrt(mean_squared_error(y_train, gbt_train_pred))
gbt_test_rmse  = np.sqrt(mean_squared_error(y_test, gbt_test_pred))
gbt_train_mae  = mean_absolute_error(y_train, gbt_train_pred)
gbt_test_mae   = mean_absolute_error(y_test, gbt_test_pred)

print(f"{'Metric':<20} {'Train':>10} {'Test':>10}")
print("-" * 42)
print(f"{'R²':<20} {gbt_train_r2:>10.4f} {gbt_test_r2:>10.4f}")
print(f"{'RMSE':<20} {gbt_train_rmse:>10.4f} {gbt_test_rmse:>10.4f}")
print(f"{'MAE':<20} {gbt_train_mae:>10.4f} {gbt_test_mae:>10.4f}")
print()

cv_scores_gbt = cross_val_score(gbt, X_train, y_train, cv=tscv, scoring='r2')
print(f"CV R² scores: {[f'{s:.4f}' for s in cv_scores_gbt]}")
print(f"Mean CV R²: {cv_scores_gbt.mean():.4f} ± {cv_scores_gbt.std():.4f}")
print()


# STEP 5: FEATURE IMPORTANCE (both types)

print("=" * 80)
print("FEATURE IMPORTANCE — IMPURITY-BASED")
print("=" * 80)

rf_imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nRandom Forest:")
for i, (feat, imp) in enumerate(rf_imp.items()):
    bar = '█' * int(imp * 50)
    print(f"  {i+1}. {feat:<40} {imp:.4f}  {bar}")

print()

gbt_imp = pd.Series(gbt.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Gradient Boosting:")
for i, (feat, imp) in enumerate(gbt_imp.items()):
    bar = '█' * int(imp * 50)
    print(f"  {i+1}. {feat:<40} {imp:.4f}  {bar}")

print()

print("=" * 80)
print("FEATURE IMPORTANCE — PERMUTATION-BASED (test set)")
print("=" * 80)

rf_perm = permutation_importance(rf, X_test, y_test, n_repeats=30, random_state=42)
rf_perm_imp = pd.Series(rf_perm.importances_mean, index=feature_cols).sort_values(ascending=False)
print("\nRandom Forest permutation:")
for i, (feat, imp) in enumerate(rf_perm_imp.items()):
    std = rf_perm.importances_std[feature_cols.index(feat)]
    print(f"  {i+1}. {feat:<40} {imp:>8.4f} ± {std:.4f}")

print()

gbt_perm = permutation_importance(gbt, X_test, y_test, n_repeats=30, random_state=42)
gbt_perm_imp = pd.Series(gbt_perm.importances_mean, index=feature_cols).sort_values(ascending=False)
print("Gradient Boosting permutation:")
for i, (feat, imp) in enumerate(gbt_perm_imp.items()):
    std = gbt_perm.importances_std[feature_cols.index(feat)]
    print(f"  {i+1}. {feat:<40} {imp:>8.4f} ± {std:.4f}")

print()


# STEP 6: PARTIAL DEPENDENCE PLOTS

best_model = rf if rf_test_r2 >= gbt_test_r2 else gbt
best_name  = "Random Forest" if rf_test_r2 >= gbt_test_r2 else "Gradient Boosting"
best_imp   = rf_imp if rf_test_r2 >= gbt_test_r2 else gbt_imp

print("=" * 80)
print(f"PARTIAL DEPENDENCE PLOTS (using {best_name})")
print("=" * 80)

top_features = best_imp.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Partial Dependence Plots — {best_name}', fontsize=16)

for idx, feat in enumerate(top_features):
    row, col = idx // 3, idx % 3
    ax = axes[row][col]

    feat_values = np.linspace(X_train[feat].min(), X_train[feat].max(), 50)
    pd_values = []

    for val in feat_values:
        X_temp = X_train.copy()
        X_temp[feat] = val
        pd_values.append(best_model.predict(X_temp).mean())

    ax.plot(feat_values, pd_values, 'b-', linewidth=2)
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel('Partial Dependence (log price)', fontsize=9)
    ax.set_title(feat.replace('log_', '').replace('_', ' '), fontsize=11)
    ax.plot(X_train[feat], [min(pd_values)] * len(X_train), '|', color='gray', alpha=0.3)

plt.tight_layout()
plt.savefig('partial_dependence_plots.png', dpi=150, bbox_inches='tight')
print("Saved: partial_dependence_plots.png")
print()


# STEP 7: PREDICTED VS ACTUAL

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, name, test_pred, train_pred in [
    (axes[0], 'Random Forest', rf_test_pred, rf_train_pred),
    (axes[1], 'Gradient Boosting', gbt_test_pred, gbt_train_pred)
]:
    ax.plot(train_df['year'], np.exp(y_train), 'b-o', label='Actual (train)', markersize=4)
    ax.plot(test_df['year'], np.exp(y_test), 'g-o', label='Actual (test)', markersize=5)
    ax.plot(train_df['year'], np.exp(train_pred), 'b--', alpha=0.5, label='Predicted (train)')
    ax.plot(test_df['year'], np.exp(test_pred), 'r-s', label='Predicted (test)', markersize=5)
    ax.axvspan(test_df['year'].min() - 0.5, test_df['year'].max() + 0.5,
               alpha=0.1, color='red', label='Test period')
    ax.set_xlabel('Year')
    ax.set_ylabel('Pistachio Price ($/lb)')
    ax.set_title(f'{name} — Predicted vs Actual')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('predicted_vs_actual.png', dpi=150, bbox_inches='tight')
print("Saved: predicted_vs_actual.png")
print()


# STEP 8: COMPARISON

print("=" * 80)
print("MODEL COMPARISON: RF vs GBT")
print("=" * 80)
print(f"{'Metric':<25} {'RF Train':>10} {'RF Test':>10} {'GBT Train':>10} {'GBT Test':>10}")
print("-" * 67)
print(f"{'R²':<25} {rf_train_r2:>10.4f} {rf_test_r2:>10.4f} {gbt_train_r2:>10.4f} {gbt_test_r2:>10.4f}")
print(f"{'RMSE':<25} {rf_train_rmse:>10.4f} {rf_test_rmse:>10.4f} {gbt_train_rmse:>10.4f} {gbt_test_rmse:>10.4f}")
print(f"{'MAE':<25} {rf_train_mae:>10.4f} {rf_test_mae:>10.4f} {gbt_train_mae:>10.4f} {gbt_test_mae:>10.4f}")
print(f"{'CV R² (mean)':<25} {cv_scores.mean():>10.4f} {'':>10} {cv_scores_gbt.mean():>10.4f}")
print()

rf_gap  = rf_train_r2 - rf_test_r2
gbt_gap = gbt_train_r2 - gbt_test_r2
if rf_test_r2 > gbt_test_r2:
    print(f"→ RF generalizes better (test R²: {rf_test_r2:.4f} vs {gbt_test_r2:.4f})")
else:
    print(f"→ GBT generalizes better (test R²: {gbt_test_r2:.4f} vs {rf_test_r2:.4f})")
print(f"→ RF overfitting gap:  {rf_gap:.4f}")
print(f"→ GBT overfitting gap: {gbt_gap:.4f}")

print()
print("=" * 80)
print("DONE")
print("=" * 80)

Features selected: 8
Features: ['log_pistachio_production_mt_lag1', 'log_wti_oil_price', 'interact_oil_fert', 'log_us_disposable_income_bn', 'log_almond_price_lb', 'fresno_rainfall_inches', 'log_pistachio_price_lb_lag1', 'biennial_dummy']

Usable observations: 44
Year range: 1981 - 2024

Train: 1981-2015 (35 obs)
Test:  2016-2024 (9 obs)
Feature-to-observation ratio: 8:35 = 1:4

RANDOM FOREST
Metric                    Train       Test
------------------------------------------
R²                       0.8523     0.1055
RMSE                     0.1414     0.1795
MAE                      0.1073     0.1620

CV R² scores: ['-3.0779', '-0.4860', '-3.9275']
Mean CV R²: -2.4971 ± 1.4637

GRADIENT BOOSTING
Metric                    Train       Test
------------------------------------------
R²                       0.9757    -0.0393
RMSE                     0.0573     0.1935
MAE                      0.0450     0.1599

CV R² scores: ['-2.6315', '-0.5901', '-2.8705']
Mean CV R²: -2.0307 ± 1.0233

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.2, 'grid.color': '#cccccc',
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 130, 'font.size': 10,
})

COLORS = {
    'rf': '#2980b9', 'gbt': '#e67e22', 'actual_train': '#2c3e50',
    'actual_test': '#27ae60', 'test_bg': '#e74c3c',
}

# =============================================================================
# 0. REQUIRES PART 1 (df must exist with all engineered features)
# =============================================================================

# =============================================================================
# 1. FEATURE MATRIX — curated 8-feature set
# =============================================================================
target = 'log_pistachio_price_lb'

feature_cols = [
    'log_pistachio_production_mt_lag1',   # domestic supply (lagged — known at decision time)
    'log_wti_oil_price',                  # input cost proxy (energy, transport)
    'interact_oil_fert',                  # combined input cost pressure
    'log_us_disposable_income_bn',        # demand-side purchasing power
    'log_almond_price_lb',                # substitute crop price
    'fresno_rainfall_inches',             # weather / drought proxy
    'log_pistachio_price_lb_lag1',        # price persistence / mean reversion
    'biennial_dummy',                     # agronomic on/off year cycle
]

feature_labels = {
    'log_pistachio_production_mt_lag1': 'US Production (lag 1)',
    'log_wti_oil_price':                'Oil Price',
    'interact_oil_fert':                'Oil × Fertilizer',
    'log_us_disposable_income_bn':      'US Disposable Income',
    'log_almond_price_lb':              'Almond Price',
    'fresno_rainfall_inches':           'Fresno Rainfall',
    'log_pistachio_price_lb_lag1':      'Price (lag 1)',
    'biennial_dummy':                   'Biennial Dummy',
}

feature_cols = [c for c in feature_cols if c in df.columns]
print(f"Features: {len(feature_cols)}")
for f in feature_cols:
    print(f"  • {feature_labels.get(f, f)}")
print()

model_df = df[['year', target] + feature_cols].dropna().reset_index(drop=True)
print(f"Usable observations: {len(model_df)} ({model_df['year'].min()}–{model_df['year'].max()})")
print()

# =============================================================================
# 2. TRAIN-TEST SPLIT — 2017–2024 test per Ruslan
# =============================================================================
train_df = model_df[model_df['year'] < 2017].copy()
test_df  = model_df[model_df['year'] >= 2017].copy()

X_train = train_df[feature_cols]
y_train = train_df[target]
X_test  = test_df[feature_cols]
y_test  = test_df[target]

print(f"Train: {train_df['year'].min()}–{train_df['year'].max()} ({len(train_df)} obs)")
print(f"Test:  {test_df['year'].min()}–{test_df['year'].max()} ({len(test_df)} obs)")
print(f"Ratio: {len(feature_cols)} features / {len(train_df)} training obs = 1:{len(train_df)//len(feature_cols)}")
print()

# =============================================================================
# 3. METRICS FUNCTION (MSE, RMSE, MAPE as Ruslan requested)
# =============================================================================
def compute_metrics(y_true_log, y_pred_log, label):
    """Compute all metrics in both log and price space."""
    # Log space
    r2_log   = r2_score(y_true_log, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))

    # Price space (for interpretable MAPE)
    actual_price = np.exp(y_true_log)
    pred_price   = np.exp(y_pred_log)
    mse   = np.mean((actual_price - pred_price) ** 2)
    rmse  = np.sqrt(mse)
    mae   = np.mean(np.abs(actual_price - pred_price))
    mape  = np.mean(np.abs((actual_price - pred_price) / actual_price)) * 100
    r2_px = 1 - np.sum((actual_price - pred_price)**2) / np.sum((actual_price - actual_price.mean())**2)

    return {
        'R² (log)': r2_log, 'RMSE (log)': rmse_log,
        'R² ($/lb)': r2_px, 'MSE ($/lb)': mse, 'RMSE ($/lb)': rmse,
        'MAE ($/lb)': mae, 'MAPE (%)': mape, 'label': label,
    }

# =============================================================================
# 4. RANDOM FOREST
# =============================================================================
print("=" * 80)
print("RANDOM FOREST")
print("=" * 80)

tscv = TimeSeriesSplit(n_splits=3)

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=3,
    min_samples_split=5,
    min_samples_leaf=4,
    max_features=0.6,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

rf_train_pred = rf.predict(X_train)
rf_test_pred  = rf.predict(X_test)

rf_train_m = compute_metrics(y_train.values, rf_train_pred, 'RF Train')
rf_test_m  = compute_metrics(y_test.values, rf_test_pred, 'RF Test')

print(f"\n{'Metric':<18} {'Train':>12} {'Test':>12}")
print("-" * 44)
for key in ['R² (log)', 'R² ($/lb)', 'MSE ($/lb)', 'RMSE ($/lb)', 'MAE ($/lb)', 'MAPE (%)']:
    fmt = '.2f' if 'MAPE' in key else '.4f'
    print(f"{key:<18} {rf_train_m[key]:>12{fmt}} {rf_test_m[key]:>12{fmt}}")

cv_scores_rf = cross_val_score(rf, X_train, y_train, cv=tscv, scoring='r2')
print(f"\nTS-CV R² scores: {[f'{s:.4f}' for s in cv_scores_rf]}")
print(f"Mean CV R²: {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}")
print()

# =============================================================================
# 5. GRADIENT BOOSTING
# =============================================================================
print("=" * 80)
print("GRADIENT BOOSTING")
print("=" * 80)

gbt = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=2,
    learning_rate=0.03,
    min_samples_split=5,
    min_samples_leaf=4,
    subsample=0.7,
    random_state=42,
)
gbt.fit(X_train, y_train)

gbt_train_pred = gbt.predict(X_train)
gbt_test_pred  = gbt.predict(X_test)

gbt_train_m = compute_metrics(y_train.values, gbt_train_pred, 'GBT Train')
gbt_test_m  = compute_metrics(y_test.values, gbt_test_pred, 'GBT Test')

print(f"\n{'Metric':<18} {'Train':>12} {'Test':>12}")
print("-" * 44)
for key in ['R² (log)', 'R² ($/lb)', 'MSE ($/lb)', 'RMSE ($/lb)', 'MAE ($/lb)', 'MAPE (%)']:
    fmt = '.2f' if 'MAPE' in key else '.4f'
    print(f"{key:<18} {gbt_train_m[key]:>12{fmt}} {gbt_test_m[key]:>12{fmt}}")

cv_scores_gbt = cross_val_score(gbt, X_train, y_train, cv=tscv, scoring='r2')
print(f"\nTS-CV R² scores: {[f'{s:.4f}' for s in cv_scores_gbt]}")
print(f"Mean CV R²: {cv_scores_gbt.mean():.4f} ± {cv_scores_gbt.std():.4f}")
print()

# =============================================================================
# 6. FEATURE IMPORTANCE — IMPURITY + PERMUTATION
# =============================================================================
print("=" * 80)
print("FEATURE IMPORTANCE")
print("=" * 80)

# Impurity-based
rf_imp  = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
gbt_imp = pd.Series(gbt.feature_importances_, index=feature_cols).sort_values(ascending=False)

print("\n── Impurity-Based ──")
print(f"\n{'Rank':<6} {'Feature':<28} {'RF':>8} {'GBT':>8}")
print("-" * 54)
all_features_ranked = rf_imp.index.tolist()
for i, feat in enumerate(all_features_ranked):
    label = feature_labels.get(feat, feat)
    print(f"{i+1:<6} {label:<28} {rf_imp[feat]:>8.4f} {gbt_imp[feat]:>8.4f}")

# Permutation-based (on test set — more reliable)
rf_perm  = permutation_importance(rf, X_test, y_test, n_repeats=30, random_state=42)
gbt_perm = permutation_importance(gbt, X_test, y_test, n_repeats=30, random_state=42)

rf_perm_imp  = pd.Series(rf_perm.importances_mean, index=feature_cols).sort_values(ascending=False)
gbt_perm_imp = pd.Series(gbt_perm.importances_mean, index=feature_cols).sort_values(ascending=False)

print("\n── Permutation-Based (test set) ──")
print(f"\n{'Rank':<6} {'Feature':<28} {'RF':>10} {'GBT':>10}")
print("-" * 58)
for i, feat in enumerate(rf_perm_imp.index):
    label = feature_labels.get(feat, feat)
    rf_val  = rf_perm_imp[feat]
    gbt_val = gbt_perm_imp[feat]
    print(f"{i+1:<6} {label:<28} {rf_val:>10.4f} {gbt_val:>10.4f}")
print()

# =============================================================================
# 7. FEATURE IMPORTANCE PLOT
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, imp, perm, name, color in [
    (axes[0], rf_imp, rf_perm_imp, 'Random Forest', COLORS['rf']),
    (axes[1], gbt_imp, gbt_perm_imp, 'Gradient Boosting', COLORS['gbt']),
]:
    labels_sorted = [feature_labels.get(f, f) for f in imp.index]

    y_pos = np.arange(len(imp))
    ax.barh(y_pos, imp.values, height=0.4, align='edge', color=color, alpha=0.7,
            label='Impurity')
    ax.barh(y_pos - 0.4, [max(0, v) for v in perm.reindex(imp.index).values],
            height=0.4, align='edge', color=color, alpha=0.35, hatch='//',
            label='Permutation')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels_sorted, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Importance')
    ax.set_title(f'{name} — Feature Importance', fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('tree_feature_importance.png', dpi=150, bbox_inches='tight')
print("Saved: tree_feature_importance.png")
print()

# =============================================================================
# 8. PARTIAL DEPENDENCE PLOTS
# =============================================================================
best_model = rf if rf_test_m['R² (log)'] >= gbt_test_m['R² (log)'] else gbt
best_name  = 'Random Forest' if rf_test_m['R² (log)'] >= gbt_test_m['R² (log)'] else 'Gradient Boosting'
best_imp   = rf_imp if best_name == 'Random Forest' else gbt_imp

top_6 = best_imp.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Partial Dependence Plots — {best_name}', fontsize=14, fontweight='bold')

for idx, feat in enumerate(top_6):
    row, col = idx // 3, idx % 3
    ax = axes[row][col]

    feat_values = np.linspace(X_train[feat].min(), X_train[feat].max(), 50)
    pd_values = []
    for val in feat_values:
        X_temp = X_train.copy()
        X_temp[feat] = val
        pd_values.append(best_model.predict(X_temp).mean())

    pd_arr = np.array(pd_values)

    # LOWESS smooth for cleaner visual
    smooth = lowess(pd_arr, feat_values, frac=0.3, return_sorted=True)
    ax.plot(smooth[:, 0], smooth[:, 1], color=COLORS['rf'], lw=2.5)
    ax.fill_between(smooth[:, 0],
                    smooth[:, 1] - 0.02,
                    smooth[:, 1] + 0.02,
                    alpha=0.15, color=COLORS['rf'])

    # Rug plot
    ax.plot(X_train[feat], [pd_arr.min() - 0.01] * len(X_train),
            '|', color='gray', alpha=0.4, markersize=8)

    label = feature_labels.get(feat, feat)
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel('Partial Dependence\n(log price)', fontsize=9)
    ax.set_title(label, fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('tree_partial_dependence.png', dpi=150, bbox_inches='tight')
print("Saved: tree_partial_dependence.png")
print()

# =============================================================================
# 9. PREDICTED VS ACTUAL — both models, professional layout
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

for ax, name, train_pred, test_pred, color in [
    (axes[0], 'Random Forest',     rf_train_pred,  rf_test_pred,  COLORS['rf']),
    (axes[1], 'Gradient Boosting', gbt_train_pred, gbt_test_pred, COLORS['gbt']),
]:
    actual_train_price = np.exp(y_train)
    actual_test_price  = np.exp(y_test)
    pred_train_price   = np.exp(train_pred)
    pred_test_price    = np.exp(test_pred)

    ax.plot(train_df['year'], actual_train_price, 'o-',
            color=COLORS['actual_train'], markersize=3, lw=1.5, label='Actual (train)')
    ax.plot(test_df['year'], actual_test_price, 'o-',
            color=COLORS['actual_test'], markersize=5, lw=1.8, label='Actual (test)')
    ax.plot(train_df['year'], pred_train_price, '--',
            color=color, alpha=0.4, lw=1.2, label='Pred (train)')
    ax.plot(test_df['year'], pred_test_price, 's-',
            color=color, markersize=6, lw=2.0, label='Pred (test)')

    ax.axvspan(2016.5, 2024.5, alpha=0.06, color=COLORS['test_bg'])
    ax.axvline(2016.5, color='gray', ls=':', lw=1)

    # Annotate test metrics
    m = rf_test_m if 'Random' in name else gbt_test_m
    ax.text(0.98, 0.02,
            f"Test RMSE: ${m['RMSE ($/lb)']:.3f}\nTest MAPE: {m['MAPE (%)']:.1f}%",
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=9, bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

    ax.set_xlabel('Year')
    ax.set_ylabel('Pistachio Price ($/lb)')
    ax.set_title(f'{name} — Predicted vs Actual', fontweight='bold')
    ax.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('tree_predicted_vs_actual.png', dpi=150, bbox_inches='tight')
print("Saved: tree_predicted_vs_actual.png")
print()

# =============================================================================
# 10. FULL MODEL COMPARISON TABLE
# =============================================================================
print("=" * 80)
print("MODEL COMPARISON — Random Forest vs Gradient Boosting")
print("=" * 80)
print(f"\n{'Metric':<18} {'RF Train':>12} {'RF Test':>12} {'GBT Train':>12} {'GBT Test':>12}")
print("-" * 70)
for key in ['R² (log)', 'R² ($/lb)', 'MSE ($/lb)', 'RMSE ($/lb)', 'MAE ($/lb)', 'MAPE (%)']:
    fmt = '.2f' if 'MAPE' in key else '.4f'
    print(f"{key:<18} {rf_train_m[key]:>12{fmt}} {rf_test_m[key]:>12{fmt}} "
          f"{gbt_train_m[key]:>12{fmt}} {gbt_test_m[key]:>12{fmt}}")
print(f"{'CV R² (mean)':<18} {cv_scores_rf.mean():>12.4f} {'':>12} "
      f"{cv_scores_gbt.mean():>12.4f}")

# Overfitting diagnosis
rf_gap  = rf_train_m['R² (log)'] - rf_test_m['R² (log)']
gbt_gap = gbt_train_m['R² (log)'] - gbt_test_m['R² (log)']
print(f"\nOverfitting gap (train R² - test R²):")
print(f"  RF:  {rf_gap:.4f}")
print(f"  GBT: {gbt_gap:.4f}")

winner = 'RF' if rf_test_m['MAPE (%)'] < gbt_test_m['MAPE (%)'] else 'GBT'
print(f"\n→ {winner} has lower test MAPE "
      f"({min(rf_test_m['MAPE (%)'], gbt_test_m['MAPE (%)']):.1f}% vs "
      f"{max(rf_test_m['MAPE (%)'], gbt_test_m['MAPE (%)']):.1f}%)")
print()

# =============================================================================
# 11. CONSENSUS DRIVERS — ARDL vs Trees
# =============================================================================
print("=" * 80)
print("CONSENSUS DRIVERS — ARDL Long-Run vs Tree Importance")
print("=" * 80)
print()
print("This table cross-references the ARDL long-run elasticities with")
print("the tree-based feature importance rankings to identify drivers")
print("that are robust across fundamentally different model families.")
print()

# Map ARDL variable names to tree feature names
ardl_to_tree = {
    'log_us_prod': 'log_pistachio_production_mt_lag1',
    'log_oil':     'log_wti_oil_price',
    'log_cpi':     None,  # CPI not directly in tree features
}

print(f"{'Driver':<28} {'ARDL LR coef':>14} {'RF Imp':>10} {'GBT Imp':>10} {'Consensus':>12}")
print("-" * 78)

consensus_drivers = []
for var in ['log_us_prod', 'log_iran', 'log_oil', 'log_cpi']:
    lr_val = globals().get('res', {}).get('lr', {}).get(var, {}).get('coef', '—')
    tree_f  = ardl_to_tree.get(var)
    rf_val  = rf_imp.get(tree_f, 0) if tree_f else '—'
    gbt_val = gbt_imp.get(tree_f, 0) if tree_f else '—'

    label = {
        'log_us_prod': 'US Production',
        'log_iran': 'Iran Production',
        'log_oil': 'Oil Price',
        'log_cpi': 'CPI',
    }[var]

    # Consensus = appears in both ARDL (significant) and tree top-5
    in_ardl = isinstance(lr_val, (int, float)) and lr_val != '—'
    in_tree = (isinstance(rf_val, float) and rf_val > 0.05) or \
              (isinstance(gbt_val, float) and gbt_val > 0.05)
    consensus = '✓' if in_ardl and in_tree else '—'

    rf_str  = f'{rf_val:.4f}' if isinstance(rf_val, float) else rf_val
    gbt_str = f'{gbt_val:.4f}' if isinstance(gbt_val, float) else gbt_val
    lr_str  = f'{lr_val:.4f}' if isinstance(lr_val, float) else str(lr_val)

    print(f"{label:<28} {lr_str:>14} {rf_str:>10} {gbt_str:>10} {consensus:>12}")

# Also show tree-only drivers
tree_only = ['interact_oil_fert', 'log_almond_price_lb', 'fresno_rainfall_inches',
             'log_pistachio_price_lb_lag1', 'biennial_dummy', 'log_us_disposable_income_bn']
for feat in tree_only:
    if feat in feature_cols:
        label = feature_labels.get(feat, feat)
        rf_val = rf_imp.get(feat, 0)
        gbt_val = gbt_imp.get(feat, 0)
        if rf_val > 0.02 or gbt_val > 0.02:
            print(f"{label:<28} {'(tree only)':>14} {rf_val:>10.4f} {gbt_val:>10.4f} {'—':>12}")

print()
print("=" * 80)
print("ALL MODELS COMPLETE — share these results + .png files with the team")
print("=" * 80)

Features: 8
  • US Production (lag 1)
  • Oil Price
  • Oil × Fertilizer
  • US Disposable Income
  • Almond Price
  • Fresno Rainfall
  • Price (lag 1)
  • Biennial Dummy

Usable observations: 44 (1981–2024)

Train: 1981–2016 (36 obs)
Test:  2017–2024 (8 obs)
Ratio: 8 features / 36 training obs = 1:4

RANDOM FOREST

Metric                    Train         Test
--------------------------------------------
R² (log)                 0.8523      -0.1074
R² ($/lb)                0.7939      -0.0857
MSE ($/lb)               0.0977       0.1688
RMSE ($/lb)              0.3125       0.4108
MAE ($/lb)               0.1891       0.3936
MAPE (%)                  10.37        18.73

TS-CV R² scores: ['-6.1609', '-0.5030', '-3.0083']
Mean CV R²: -3.2241 ± 2.3149

GRADIENT BOOSTING

Metric                    Train         Test
--------------------------------------------
R² (log)                 0.9766      -0.3062
R² ($/lb)                0.9815      -0.2459
MSE ($/lb)               0.0087       0.

NameError: name 'res' is not defined